# Wrangle a small single-cell cohort

This notebook follows a realistic analysis boundary: start with an annotated AnnData object, attach a sample sheet, apply cell-level QC, derive a marker score from a count layer, rank cells within samples, and produce both an AnnData result and a cohort summary. The data are synthetic and intentionally small, so the notebook is self-contained and every intermediate result is easy to inspect.

## Create a PBMC-like AnnData object

The object uses normalized values in `X`, raw counts in `layers["counts"]`, cell annotations in `obs`, and gene annotations in `var`—a common structure after basic Scanpy preprocessing.

In [ ]:
import anndata as ad
import annplyr as ap
import numpy as np
import pandas as pd
from scipy import sparse

counts = np.array(
    [
        [8, 4, 2, 1, 0, 0],
        [0, 0, 7, 5, 1, 0],
        [6, 5, 1, 0, 0, 0],
        [0, 0, 3, 4, 6, 2],
        [9, 6, 1, 0, 0, 0],
        [0, 0, 2, 6, 1, 0],
        [1, 0, 1, 0, 7, 5],
        [3, 2, 5, 4, 0, 0],
    ],
    dtype=np.int64,
)

obs = pd.DataFrame(
    {
        "sample_id": pd.Categorical(["S1", "S1", "S2", "S2", "S3", "S3", "S2", "S3"]),
        "cell_type": pd.Categorical(
            ["B cell", "T cell", "B cell", "Monocyte", "B cell", "T cell", "Monocyte", "T cell"]
        ),
        "total_counts": pd.array([1200, 900, 1500, 1100, 1800, 800, 1300, 1600], dtype="Int64"),
        "pct_counts_mt": np.array([4.0, 12.0, 3.0, 7.5, 6.0, 18.0, 9.0, 2.0], dtype=np.float32),
    },
    index=[f"cell_{i}" for i in range(8)],
)
var = pd.DataFrame(
    {"feature_type": pd.Categorical(["gene"] * 6)},
    index=["MS4A1", "CD79A", "CD3D", "NKG7", "LST1", "FCGR3A"],
)

adata = ad.AnnData(X=sparse.csr_matrix(np.log1p(counts)), obs=obs, var=var)
adata.layers["counts"] = sparse.csr_matrix(counts)
adata

## Attach experimental metadata

The sample sheet has one row per sample, whereas `obs` has one row per cell. Declaring `relationship="many-to-one"` makes that assumption executable. A duplicated sample-sheet key would raise an error instead of duplicating cells.

In [ ]:
sample_sheet = pd.DataFrame(
    {
        "sample_id": ["S1", "S2", "S3"],
        "condition": ["control", "stimulated", "control"],
        "donor_age": [34, 51, 42],
    }
)

cohort = adata.ap.left_join(
    sample_sheet,
    by="sample_id",
    relationship="many-to-one",
)
cohort.obs.head()

## Apply the QC rule

A tuple of predicates is an implicit `AND`. The returned AnnData is independent and all aligned containers use the same cell positions.

In [ ]:
qc = cohort.ap.filter(
    obs=(
        ap.col("total_counts") >= 1_000,
        ap.col("pct_counts_mt") < 10,
    )
)
qc.obs[["sample_id", "condition", "cell_type", "total_counts", "pct_counts_mt"]]

## Derive a marker score from the count layer

Matrix expressions are read-only inputs to `mutate()`. `layer="counts"` selects the count layer, while the new score is written to `obs`. The budget states that exactly two genes per retained cell may be read.

In [ ]:
scored = qc.ap.mutate(
    x={"B_cell_score": (ap.col("MS4A1") + ap.col("CD79A")) / 2},
    layer="counts",
    max_matrix_values=2 * qc.n_obs,
)
scored.obs[["cell_type", "B_cell_score"]]

## Rank within samples and summarize the cohort

Persistent grouping is useful when several operations share a comparison unit. The first result remains grouped until `ungroup()`; the second collapses cells to a pandas summary table.

In [ ]:
ranked = scored.ap.group_by(obs="sample_id").mutate(
    obs={"within_sample_rank": ap.min_rank("total_counts", descending=True)}
)
ranked.ungroup().obs[["sample_id", "total_counts", "within_sample_rank"]]

In [ ]:
cohort_summary = ranked.ungroup().ap.summarize(
    obs={
        "cells": ap.n(),
        "mean_counts": ap.mean("total_counts"),
        "mean_B_cell_score": ap.mean("B_cell_score"),
    },
    x={"mean_MS4A1": ap.mean("MS4A1")},
    by=["condition", "cell_type"],
    layer="counts",
)
cohort_summary

## Create focused analysis and plotting outputs

Keep the scientific object as AnnData, then materialize only the genes needed by the figure. The two outputs can move independently: `analysis` to another scverse method and `plot_data` to a plotting library.

In [ ]:
analysis = ranked.ungroup().ap.select(
    obs=["sample_id", "condition", "cell_type", "B_cell_score", "within_sample_rank"],
    x=["MS4A1", "CD79A", "CD3D", "LST1"],
)
analysis

In [ ]:
plot_data = analysis.ap.to_tidy(
    obs=["sample_id", "condition", "cell_type"],
    x=["MS4A1", "CD79A", "CD3D"],
    layer="counts",
    max_matrix_values=3 * analysis.n_obs,
)
plot_data.head(9)

## Takeaway

The workflow never manually synchronizes a pandas mask with `X`, layers, or embeddings. AnnData-returning verbs keep those relationships intact; joins check sample-sheet assumptions; grouped verbs express the unit of comparison; and table extraction is postponed until a pandas result is genuinely useful.